# 02 — LoRA Fine-Tuning
Train a LoRA adapter on FOLIO using the config-driven `scripts/train.py` approach.
Config is loaded from `configs/lora/phi35.yml` — change this to switch models.

In [ ]:
# ── Colab setup ────────────────────────────────────────────────────────────
import os
from google.colab import drive, userdata   # ← add userdata here
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    # Update this path if your outputs folder is elsewhere
    OUTPUT_BASE = '/content/drive/MyDrive/slm_logic_hardening/outputs'
    !pip install -q transformers datasets pyyaml sentencepiece
    # Navigate directly to your existing project folder in Drive
    %cd '/content/drive/MyDrive/slm_logic_hardening'
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
else:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path().resolve()))
    OUTPUT_BASE = 'outputs'

os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

Mounted at /content/drive
/content/drive/MyDrive/slm_logic_hardening


In [ ]:
import torch
device = 'mps' if torch.backends.mps.is_available() else \
         'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 1. Load config

In [ ]:
import yaml

#CONFIG_PATH = 'configs/lora/phi35.yml'
CONFIG_PATH = 'configs/lora/phi35_folio_pw.yml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('Model key:', cfg['model_key'])
print('Experiment:', cfg['experiment_name'])
print('LoRA rank:', cfg['lora']['r'])
print('Learning rate:', cfg['training']['learning_rate'])
print('Epochs:', cfg['training']['num_train_epochs'])
print(cfg.get('dataset_sample_sizes'))  # should show {'proofwriter': 2000}

Model key: phi35
Experiment: phi35_lora_folio_pw
LoRA rank: 64
Learning rate: 0.0001
Epochs: 5
{'proofwriter': 2000}


## 2. Load model and apply LoRA

In [ ]:
from models.model_loader import load_model
from peft import LoraConfig, get_peft_model

model, tokenizer, device = load_model(cfg['model_key'], mode='baseline')

lc = cfg['lora']
lora_config = LoraConfig(
    r=lc['r'],
    lora_alpha=lc['lora_alpha'],
    target_modules=lc['target_modules'],
    lora_dropout=lc['lora_dropout'],
    bias=lc['bias'],
    task_type=lc['task_type'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282


## 3. Prepare datasets

In [ ]:
from data.load_data import load_datasets_for_training
from data.preprocess import DATASET_PREPROCESSORS
from scripts.train import build_tokenize_fn
from datasets import concatenate_datasets

dataset_names = cfg.get('datasets', ['folio'])
sample_sizes = cfg.get('dataset_sample_sizes', {})
raw_datasets = load_datasets_for_training(dataset_names, sample_sizes=sample_sizes)

max_length = cfg['training']['max_length']
tokenize_fn = build_tokenize_fn(tokenizer, max_length)
columns_to_keep = ['input_ids', 'attention_mask', 'labels']

all_train = []
folio_val = None  # early-stopping validation — carved from FOLIO train split

for name, raw in raw_datasets.items():
    preprocessor = DATASET_PREPROCESSORS[name]
    processed = preprocessor(raw)

    if name == 'folio':
        # Split FOLIO train into 850 train / 151 validation so the 203-sample
        # validation set stays unseen during training and gives unbiased eval
        folio_split = processed['train'].train_test_split(test_size=151, seed=42)
        t = folio_split['train'].map(tokenize_fn, load_from_cache_file=False)
        t = t.remove_columns([c for c in t.column_names if c not in columns_to_keep])

        v = folio_split['test'].map(tokenize_fn, load_from_cache_file=False)
        v = v.remove_columns([c for c in v.column_names if c not in columns_to_keep])
        folio_val = v
    else:
        t = processed['train'].map(tokenize_fn, load_from_cache_file=False)
        t = t.remove_columns([c for c in t.column_names if c not in columns_to_keep])

    all_train.append(t)

train_dataset = concatenate_datasets(all_train) if len(all_train) > 1 else all_train[0]
val_dataset = folio_val

print(f'Datasets: {dataset_names}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} (FOLIO train split, unseen during eval)')

## 4. Train

In [ ]:
from pathlib import Path
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

tc = cfg['training']
output_dir = Path(OUTPUT_BASE) / cfg['model_key'] / cfg['experiment_name']
output_dir.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=tc['num_train_epochs'],
    per_device_train_batch_size=tc['per_device_train_batch_size'],
    per_device_eval_batch_size=tc['per_device_eval_batch_size'],
    gradient_accumulation_steps=tc.get('gradient_accumulation_steps', 1),
    learning_rate=tc['learning_rate'],
    warmup_steps=tc['warmup_steps'],
    max_grad_norm=tc['max_grad_norm'],
    logging_steps=tc['logging_steps'],
    eval_strategy=tc['eval_strategy'],
    save_strategy=tc['save_strategy'],
    fp16=tc['fp16'],
    bf16=tc['bf16'],
    load_best_model_at_end=tc['load_best_model_at_end'],
    metric_for_best_model=tc['metric_for_best_model'],
    report_to=tc['report_to'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 5. Save adapter

In [ ]:
adapter_path = output_dir / 'final_adapter'
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
print(f'Adapter saved to {adapter_path}')

Adapter saved to /content/drive/MyDrive/slm_logic_hardening/outputs/phi35/phi35_lora_folio_pw/final_adapter
